# Gaming & Mental Health — Path Analysis
Decomposes the causal chain: **gaming habits → lifestyle mediators → addiction risk**

Uses sequential OLS (statsmodels) to estimate standardised path coefficients,
then visualises the weighted DAG and decomposes direct vs. indirect effects.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

plt.rcParams.update({
    'figure.dpi': 150,
    'axes.facecolor': '#0d0d0d',
    'figure.facecolor': '#0d0d0d',
    'axes.edgecolor': '#444',
    'axes.labelcolor': '#ccc',
    'xtick.color': '#aaa',
    'ytick.color': '#aaa',
    'text.color': '#eee',
    'grid.color': '#333',
    'grid.linestyle': '--',
})

DATA_PATH = '../data/Gaming and Mental Health.csv'
SEED = 42
np.random.seed(SEED)

## 1. Data Loading & EDA

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head(3)

: 

: 

In [ ]:
# Distribution of key numeric variables
numeric_cols = ['daily_gaming_hours', 'sleep_hours', 'social_isolation_score',
                'exercise_hours_weekly', 'face_to_face_social_hours_weekly',
                'years_gaming', 'age', 'monthly_game_spending_usd']

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
fig.suptitle('Key Numeric Variable Distributions', fontsize=13, color='white', y=1.02)
for ax, col in zip(axes.flat, numeric_cols):
    ax.hist(df[col].dropna(), bins=30, color='#888', edgecolor='#333', linewidth=0.4)
    ax.set_title(col.replace('_', ' '), fontsize=8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

: 

: 

In [ ]:
# Addiction risk level distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['gaming_addiction_risk_level', 'game_genre', 'mood_state']):
    counts = df[col].value_counts()
    ax.barh(counts.index, counts.values, color='#777', edgecolor='#333')
    ax.set_title(col.replace('_', ' '), fontsize=9)
    ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

: 

: 

## 2. Preprocessing

In [ ]:
dfc = df.copy()

# Drop high-cardinality / irrelevant columns
dfc.drop(columns=['record_id', 'primary_game'], inplace=True)

# ── Boolean strings → int ────────────────────────────────────────────────────
bool_cols = ['withdrawal_symptoms', 'loss_of_other_interests',
             'continued_despite_problems', 'eye_strain', 'back_neck_pain']
for c in bool_cols:
    dfc[c] = dfc[c].map({'TRUE': 1, 'True': 1, True: 1,
                         'FALSE': 0, 'False': 0, False: 0}).astype(float)

# ── Ordinal encodings ────────────────────────────────────────────────────────
dfc['sleep_quality'] = dfc['sleep_quality'].map({
    'Insomnia': 0, 'Very Poor': 1, 'Poor': 2, 'Fair': 3, 'Good': 4
})

dfc['sleep_disruption_frequency'] = dfc['sleep_disruption_frequency'].map({
    'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Often': 3, 'Always': 4
})

# mood_state: map to negative/neutral/positive valence (0/1/2)
mood_map = {
    'Normal': 1, 'Euphoric': 2, 'Excited': 2,
    'Irritable': 0, 'Anxious': 0, 'Angry': 0,
    'Restless': 0, 'Depressed': 0, 'Withdrawn': 0
}
dfc['mood_state'] = dfc['mood_state'].map(mood_map)

dfc['mood_swing_frequency'] = dfc['mood_swing_frequency'].map({
    'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Often': 3, 'Daily': 4
})

dfc['academic_work_performance'] = dfc['academic_work_performance'].map({
    'Failing': 0, 'Poor': 1, 'Below Average': 2,
    'Average': 3, 'Good': 4, 'Excellent': 5
})

dfc['gaming_addiction_risk_level'] = dfc['gaming_addiction_risk_level'].map({
    'Low': 0, 'Moderate': 1, 'High': 2, 'Severe': 3
})

dfc['gender'] = dfc['gender'].map({'Male': 0, 'Female': 1}).fillna(0.5)

# ── Nominal → dummies ────────────────────────────────────────────────────────
dfc = pd.get_dummies(dfc, columns=['game_genre', 'gaming_platform'], drop_first=True)

# ── Missing value imputation ─────────────────────────────────────────────────
# grades_gpa: impute within academic_work_performance group
dfc['grades_gpa'] = dfc.groupby('academic_work_performance')['grades_gpa'].transform(
    lambda x: x.fillna(x.median())
)
# fallback for any remaining NaN
dfc['grades_gpa'].fillna(dfc['grades_gpa'].median(), inplace=True)

# work_productivity_score: impute within addiction risk group
dfc['work_productivity_score'] = dfc.groupby('gaming_addiction_risk_level')['work_productivity_score'].transform(
    lambda x: x.fillna(x.median())
)
dfc['work_productivity_score'].fillna(dfc['work_productivity_score'].median(), inplace=True)

assert dfc.isnull().sum().sum() == 0, 'Still have NaN after imputation!'
print(f'Preprocessed shape: {dfc.shape}')
print('No missing values ✓')

: 

: 

In [ ]:
# ── Z-score all numeric columns ───────────────────────────────────────────────
# (standardised coefficients make path weights directly comparable)
scaler = StandardScaler()
df_z = pd.DataFrame(scaler.fit_transform(dfc), columns=dfc.columns)

# ── Correlation heatmap (key variables only) ──────────────────────────────────
core = ['daily_gaming_hours', 'sleep_hours', 'sleep_quality', 'social_isolation_score',
        'exercise_hours_weekly', 'face_to_face_social_hours_weekly', 'mood_state',
        'withdrawal_symptoms', 'loss_of_other_interests', 'continued_despite_problems',
        'gaming_addiction_risk_level', 'academic_work_performance', 'work_productivity_score']

corr = df_z[core].corr()
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr, cmap='RdGy_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(core)))
ax.set_yticks(range(len(core)))
ax.set_xticklabels([c.replace('_', '\n') for c in core], fontsize=6, rotation=45, ha='right')
ax.set_yticklabels([c.replace('_', ' ') for c in core], fontsize=6)
for i in range(len(core)):
    for j in range(len(core)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
                fontsize=5, color='white' if abs(corr.iloc[i, j]) > 0.4 else '#aaa')
plt.colorbar(im, ax=ax, fraction=0.03)
ax.set_title('Correlation Matrix — Core Variables', fontsize=11)
plt.tight_layout()
plt.show()

: 

: 

## 3. DAG Structure (pre-fit)

In [ ]:
# ── Build the DAG ─────────────────────────────────────────────────────────────
EDGES = [
    # Exogenous → exposure
    ('age', 'daily_gaming_hours'),
    ('age', 'years_gaming'),
    ('years_gaming', 'daily_gaming_hours'),
    # Exposure → mediators
    ('daily_gaming_hours', 'sleep_hours'),
    ('daily_gaming_hours', 'sleep_quality'),
    ('daily_gaming_hours', 'social_isolation_score'),
    ('daily_gaming_hours', 'exercise_hours_weekly'),
    ('daily_gaming_hours', 'face_to_face_social_hours_weekly'),
    # Mediators → mood
    ('sleep_hours', 'mood_state'),
    ('sleep_quality', 'mood_state'),
    ('social_isolation_score', 'mood_state'),
    ('exercise_hours_weekly', 'mood_state'),
    ('face_to_face_social_hours_weekly', 'mood_state'),
    # → primary outcome
    ('daily_gaming_hours', 'gaming_addiction_risk_level'),
    ('sleep_hours', 'gaming_addiction_risk_level'),
    ('sleep_quality', 'gaming_addiction_risk_level'),
    ('social_isolation_score', 'gaming_addiction_risk_level'),
    ('mood_state', 'gaming_addiction_risk_level'),
    ('exercise_hours_weekly', 'gaming_addiction_risk_level'),
    ('withdrawal_symptoms', 'gaming_addiction_risk_level'),
    ('loss_of_other_interests', 'gaming_addiction_risk_level'),
    ('continued_despite_problems', 'gaming_addiction_risk_level'),
    # → secondary outcomes
    ('daily_gaming_hours', 'academic_work_performance'),
    ('mood_state', 'academic_work_performance'),
    ('sleep_hours', 'academic_work_performance'),
    ('daily_gaming_hours', 'work_productivity_score'),
    ('mood_state', 'work_productivity_score'),
    ('sleep_hours', 'work_productivity_score'),
]

G = nx.DiGraph()
G.add_edges_from(EDGES)

# Node positions — 4 horizontal layers
POS = {
    # Layer 1 — Exogenous
    'age': (-2, 3), 'years_gaming': (0, 3), 'gender': (2, 3),
    # Layer 2 — Central exposure
    'daily_gaming_hours': (0, 2),
    # Layer 3 — Mediators
    'sleep_hours': (-5, 1), 'sleep_quality': (-3, 1),
    'social_isolation_score': (-1, 1),
    'exercise_hours_weekly': (1, 1),
    'face_to_face_social_hours_weekly': (3, 1),
    # Addiction indicators
    'withdrawal_symptoms': (5.5, 1),
    'loss_of_other_interests': (7, 1),
    'continued_despite_problems': (8.5, 1),
    # Layer 3.5 — Mood synthesis
    'mood_state': (-1, 0),
    # Layer 4 — Outcomes
    'gaming_addiction_risk_level': (-3, -1),
    'academic_work_performance': (0.5, -1),
    'work_productivity_score': (3.5, -1),
}

# Node colour by role
NODE_COLORS = {
    'age': '#4466aa', 'years_gaming': '#4466aa', 'gender': '#4466aa',
    'daily_gaming_hours': '#aa6644',
    'sleep_hours': '#66aa66', 'sleep_quality': '#66aa66',
    'social_isolation_score': '#66aa66', 'exercise_hours_weekly': '#66aa66',
    'face_to_face_social_hours_weekly': '#66aa66',
    'withdrawal_symptoms': '#aa9944', 'loss_of_other_interests': '#aa9944',
    'continued_despite_problems': '#aa9944',
    'mood_state': '#9966aa',
    'gaming_addiction_risk_level': '#cc4444',
    'academic_work_performance': '#cc4444',
    'work_productivity_score': '#cc4444',
}

fig, ax = plt.subplots(figsize=(18, 10))
fig.patch.set_facecolor('#0d0d0d')
ax.set_facecolor('#0d0d0d')

node_colors = [NODE_COLORS.get(n, '#888888') for n in G.nodes()]
nx.draw_networkx(
    G, pos=POS, ax=ax,
    node_color=node_colors, node_size=1000,
    font_size=6, font_color='white',
    edge_color='#666', arrows=True,
    arrowsize=12, width=1.2,
    connectionstyle='arc3,rad=0.05'
)

# Legend
legend = [
    mpatches.Patch(color='#4466aa', label='Exogenous'),
    mpatches.Patch(color='#aa6644', label='Gaming exposure'),
    mpatches.Patch(color='#66aa66', label='Lifestyle mediators'),
    mpatches.Patch(color='#9966aa', label='Mood synthesis'),
    mpatches.Patch(color='#aa9944', label='Addiction indicators'),
    mpatches.Patch(color='#cc4444', label='Outcomes'),
]
ax.legend(handles=legend, loc='lower left', fontsize=8,
          facecolor='#1a1a1a', edgecolor='#555', labelcolor='white')
ax.set_title('Path Model DAG (pre-fit)', color='white', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

: 

: 

## 4. Model Fitting — Sequential OLS

In [ ]:
# ── Equations in topological order ───────────────────────────────────────────
# Each tuple: (outcome, [predictors])
EQUATIONS = [
    # Layer 2
    ('daily_gaming_hours',
     ['age', 'years_gaming', 'gender']),
    # Layer 3 mediators
    ('sleep_hours',
     ['daily_gaming_hours', 'age', 'gender']),
    ('sleep_quality',
     ['daily_gaming_hours', 'age', 'gender']),
    ('social_isolation_score',
     ['daily_gaming_hours', 'age', 'gender']),
    ('exercise_hours_weekly',
     ['daily_gaming_hours', 'age', 'gender']),
    ('face_to_face_social_hours_weekly',
     ['daily_gaming_hours', 'age', 'gender']),
    # Layer 3.5 — mood
    ('mood_state',
     ['sleep_hours', 'sleep_quality', 'social_isolation_score',
      'exercise_hours_weekly', 'face_to_face_social_hours_weekly', 'daily_gaming_hours']),
    # Layer 4 — primary outcome
    ('gaming_addiction_risk_level',
     ['daily_gaming_hours', 'sleep_hours', 'sleep_quality', 'social_isolation_score',
      'mood_state', 'exercise_hours_weekly', 'face_to_face_social_hours_weekly',
      'withdrawal_symptoms', 'loss_of_other_interests', 'continued_despite_problems']),
    # Secondary outcomes
    ('academic_work_performance',
     ['daily_gaming_hours', 'mood_state', 'sleep_hours', 'social_isolation_score']),
    ('work_productivity_score',
     ['daily_gaming_hours', 'mood_state', 'sleep_hours']),
]

results = {}  # outcome → statsmodels RegressionResultsWrapper

for outcome, predictors in EQUATIONS:
    X = sm.add_constant(df_z[predictors])
    y = df_z[outcome]
    model = sm.OLS(y, X).fit()
    results[outcome] = model

print('Equations fitted:')
for outcome, model in results.items():
    print(f'  {outcome:<40} R²={model.rsquared:.3f}  adj-R²={model.rsquared_adj:.3f}')

: 

: 

In [ ]:
# ── Compile path coefficient table ───────────────────────────────────────────
rows = []
for outcome, model in results.items():
    for pred, coef in model.params.items():
        if pred == 'const':
            continue
        pval = model.pvalues[pred]
        se = model.bse[pred]
        rows.append({'from': pred, 'to': outcome, 'coef': coef, 'se': se, 'pvalue': pval})

path_df = pd.DataFrame(rows)
path_df['sig'] = path_df['pvalue'].apply(lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else '')))

print(path_df[path_df['to'] == 'gaming_addiction_risk_level']
      .sort_values('coef', key=abs, ascending=False)
      [['from', 'coef', 'se', 'pvalue', 'sig']]
      .to_string(index=False))

: 

: 

In [ ]:
# ── VIF check for addiction risk equation ────────────────────────────────────
addiction_preds = EQUATIONS[7][1]  # index 7 = gaming_addiction_risk_level
X_check = df_z[addiction_preds].assign(const=1)
vif_data = pd.DataFrame({
    'feature': addiction_preds,
    'VIF': [variance_inflation_factor(X_check.values, i) for i in range(len(addiction_preds))]
})
print('VIF for addiction risk equation:')
print(vif_data.sort_values('VIF', ascending=False).to_string(index=False))
if (vif_data['VIF'] > 10).any():
    print('\n⚠ VIF > 10 detected — consider composite variables')
else:
    print('\nNo severe multicollinearity ✓')

: 

: 

## 5. Direct vs. Indirect Effects of Daily Gaming Hours → Addiction Risk

In [ ]:
# Helper: get coefficient for a (from, to) pair
def get_coef(from_var, to_var):
    row = path_df[(path_df['from'] == from_var) & (path_df['to'] == to_var)]
    return float(row['coef'].values[0]) if len(row) else 0.0

# Direct effect
direct = get_coef('daily_gaming_hours', 'gaming_addiction_risk_level')

# Single-step indirect effects (gaming → mediator → addiction)
mediators = ['sleep_hours', 'sleep_quality', 'social_isolation_score',
             'exercise_hours_weekly', 'face_to_face_social_hours_weekly']

indirect_1step = {
    m: get_coef('daily_gaming_hours', m) * get_coef(m, 'gaming_addiction_risk_level')
    for m in mediators
}

# Two-step indirect via mood_state (gaming → mediator → mood → addiction)
indirect_2step = {
    f'{m}→mood': get_coef('daily_gaming_hours', m) * get_coef(m, 'mood_state') * get_coef('mood_state', 'gaming_addiction_risk_level')
    for m in mediators
}

# Direct via mood (gaming → mood → addiction)
indirect_mood_direct = get_coef('daily_gaming_hours', 'mood_state') * get_coef('mood_state', 'gaming_addiction_risk_level')

all_indirect = {**indirect_1step, **indirect_2step, 'via_mood_direct': indirect_mood_direct}
total_indirect = sum(all_indirect.values())
total_effect = direct + total_indirect

print(f'Direct effect:         {direct:+.4f}')
print(f'Total indirect effect: {total_indirect:+.4f}')
print(f'Total effect:          {total_effect:+.4f}')
print(f'\nIndirect paths:')
for k, v in sorted(all_indirect.items(), key=lambda x: abs(x[1]), reverse=True):
    pct = v / total_effect * 100 if total_effect != 0 else 0
    print(f'  {k:<40} {v:+.4f}  ({pct:+.1f}% of total)')

: 

: 

In [ ]:
# ── Bootstrap 95% CI for indirect effects ────────────────────────────────────
N_BOOT = 500
n = len(df_z)

boot_indirect = {k: [] for k in all_indirect}
boot_direct = []

for _ in range(N_BOOT):
    idx = np.random.choice(n, n, replace=True)
    df_b = df_z.iloc[idx].reset_index(drop=True)

    # Fit equations needed for decomposition
    boot_coef = {}
    equations_needed = [
        ('daily_gaming_hours', ['age', 'years_gaming', 'gender']),
    ] + [
        (m, ['daily_gaming_hours', 'age', 'gender']) for m in mediators
    ] + [
        ('mood_state', ['sleep_hours', 'sleep_quality', 'social_isolation_score',
                        'exercise_hours_weekly', 'face_to_face_social_hours_weekly', 'daily_gaming_hours']),
        ('gaming_addiction_risk_level',
         ['daily_gaming_hours', 'sleep_hours', 'sleep_quality', 'social_isolation_score',
          'mood_state', 'exercise_hours_weekly', 'face_to_face_social_hours_weekly',
          'withdrawal_symptoms', 'loss_of_other_interests', 'continued_despite_problems'])
    ]
    for outcome, preds in equations_needed:
        Xb = sm.add_constant(df_b[preds])
        yb = df_b[outcome]
        res_b = sm.OLS(yb, Xb).fit()
        for p in preds:
            boot_coef[(p, outcome)] = res_b.params.get(p, 0.0)

    def bc(f, t):
        return boot_coef.get((f, t), 0.0)

    boot_direct.append(bc('daily_gaming_hours', 'gaming_addiction_risk_level'))
    for m in mediators:
        boot_indirect[m].append(bc('daily_gaming_hours', m) * bc(m, 'gaming_addiction_risk_level'))
        boot_indirect[f'{m}→mood'].append(bc('daily_gaming_hours', m) * bc(m, 'mood_state') * bc('mood_state', 'gaming_addiction_risk_level'))
    boot_indirect['via_mood_direct'].append(bc('daily_gaming_hours', 'mood_state') * bc('mood_state', 'gaming_addiction_risk_level'))

# Summary table
print(f'{"Pathway":<40} {"Estimate":>10} {"95% CI":>22}')
print('-' * 74)
print(f'{"DIRECT effect":<40} {direct:>+10.4f}  [{np.percentile(boot_direct,2.5):+.4f}, {np.percentile(boot_direct,97.5):+.4f}]')
for k, v in sorted(all_indirect.items(), key=lambda x: abs(x[1]), reverse=True):
    ci_lo = np.percentile(boot_indirect[k], 2.5)
    ci_hi = np.percentile(boot_indirect[k], 97.5)
    print(f'  via {k:<36} {v:>+10.4f}  [{ci_lo:+.4f}, {ci_hi:+.4f}]')
print('-' * 74)
print(f'{"TOTAL INDIRECT":<40} {total_indirect:>+10.4f}')
print(f'{"TOTAL EFFECT":<40} {total_effect:>+10.4f}')

: 

: 

## 6. Weighted Path Diagram

In [ ]:
# Prune insignificant / tiny paths
sig_edges = path_df[(path_df['pvalue'] < 0.10) & (path_df['coef'].abs() > 0.05)]

G_w = nx.DiGraph()
for _, row in sig_edges.iterrows():
    if row['from'] in POS and row['to'] in POS:
        G_w.add_edge(row['from'], row['to'],
                     weight=row['coef'],
                     pvalue=row['pvalue'])

fig, ax = plt.subplots(figsize=(20, 11))
fig.patch.set_facecolor('#0d0d0d')
ax.set_facecolor('#0d0d0d')

pos_draw = {n: p for n, p in POS.items() if n in G_w.nodes()}
node_c = [NODE_COLORS.get(n, '#888888') for n in G_w.nodes()]

# Draw nodes
nx.draw_networkx_nodes(G_w, pos_draw, ax=ax, node_color=node_c, node_size=900, alpha=0.9)
nx.draw_networkx_labels(G_w, pos_draw, ax=ax, font_size=6, font_color='white')

# Draw edges coloured by sign, width by magnitude
for u, v, d in G_w.edges(data=True):
    w = d['weight']
    color = '#44cc44' if w > 0 else '#cc4444'
    width = min(abs(w) * 6, 5)
    nx.draw_networkx_edges(
        G_w, pos_draw, edgelist=[(u, v)], ax=ax,
        edge_color=color, width=width, alpha=0.75,
        arrows=True, arrowsize=10,
        connectionstyle='arc3,rad=0.05'
    )

# Edge labels
edge_labels = {(u, v): f"{d['weight']:+.2f}" for u, v, d in G_w.edges(data=True)}
nx.draw_networkx_edge_labels(
    G_w, pos_draw, edge_labels=edge_labels, ax=ax,
    font_size=5, font_color='#ddd',
    bbox=dict(boxstyle='round,pad=0.1', fc='#111', ec='none', alpha=0.7)
)

legend = [
    mpatches.Patch(color='#44cc44', label='Positive path'),
    mpatches.Patch(color='#cc4444', label='Negative path'),
    mpatches.Patch(color='#4466aa', label='Exogenous'),
    mpatches.Patch(color='#aa6644', label='Gaming exposure'),
    mpatches.Patch(color='#66aa66', label='Lifestyle mediators'),
    mpatches.Patch(color='#9966aa', label='Mood synthesis'),
    mpatches.Patch(color='#cc4444', label='Outcomes'),
]
ax.legend(handles=legend, loc='lower left', fontsize=7,
          facecolor='#1a1a1a', edgecolor='#555', labelcolor='white')
ax.set_title('Weighted Path Diagram (standardised OLS coefficients)\nEdge width ∝ |β|  |  Green = positive, Red = negative  |  p < 0.10 only',
             color='white', fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.savefig('path_diagram_weighted.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('Saved path_diagram_weighted.png')

: 

: 

## 7. Fit Statistics

In [ ]:
fit_rows = []
for outcome, model in results.items():
    fit_rows.append({
        'Outcome': outcome,
        'R²': round(model.rsquared, 3),
        'Adj R²': round(model.rsquared_adj, 3),
        'F-stat': round(model.fvalue, 2),
        'F p-value': f'{model.f_pvalue:.2e}',
        'n': int(model.nobs),
    })

fit_table = pd.DataFrame(fit_rows).set_index('Outcome')
print(fit_table.to_string())

: 

: 

In [ ]:
# Residual plots for the 3 outcome equations
outcome_eqs = ['gaming_addiction_risk_level', 'academic_work_performance', 'work_productivity_score']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, outcome in zip(axes, outcome_eqs):
    model = results[outcome]
    fitted = model.fittedvalues
    resid = model.resid
    ax.scatter(fitted, resid, s=4, alpha=0.4, color='#888')
    ax.axhline(0, color='white', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Fitted', fontsize=8)
    ax.set_ylabel('Residual', fontsize=8)
    ax.set_title(outcome.replace('_', ' '), fontsize=9)
    ax.grid(True, alpha=0.3)
plt.suptitle('Residual vs. Fitted Plots', fontsize=11, color='white', y=1.02)
plt.tight_layout()
plt.show()

: 

: 

## 8. Key Findings Summary

In [ ]:
# Top predictors of gaming addiction risk
addiction_paths = path_df[path_df['to'] == 'gaming_addiction_risk_level'].copy()
addiction_paths = addiction_paths.sort_values('coef', key=abs, ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Gaming & Mental Health — Path Analysis Key Findings', fontsize=13, color='white', y=1.02)

# ── Plot 1: Top predictors of addiction risk ──────────────────────────────────
ax = axes[0]
colors = ['#44cc44' if c > 0 else '#cc4444' for c in addiction_paths['coef']]
bars = ax.barh(addiction_paths['from'].str.replace('_', ' '), addiction_paths['coef'],
               color=colors, edgecolor='#333', height=0.6)
ax.axvline(0, color='white', linewidth=0.8, linestyle='--')
ax.set_xlabel('Standardised coefficient (β)', fontsize=9)
ax.set_title('Predictors of Gaming Addiction Risk Level', fontsize=10)
ax.grid(True, alpha=0.3, axis='x')
for bar, sig in zip(bars, addiction_paths['sig']):
    if sig:
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                sig, va='center', fontsize=8, color='white')

# ── Plot 2: Direct vs. indirect effects of daily gaming hours ─────────────────
ax2 = axes[1]
labels = ['Direct\neffect'] + [k.replace('_', ' ') for k in all_indirect.keys()]
values = [direct] + list(all_indirect.values())
colors2 = ['#6688cc'] + ['#44cc44' if v > 0 else '#cc4444' for v in all_indirect.values()]

sorted_pairs = sorted(zip(labels, values, colors2), key=lambda x: abs(x[1]), reverse=True)
labels_s, values_s, colors_s = zip(*sorted_pairs)

ax2.barh(labels_s, values_s, color=colors_s, edgecolor='#333', height=0.6)
ax2.axvline(0, color='white', linewidth=0.8, linestyle='--')
ax2.set_xlabel('Effect size (standardised β)', fontsize=9)
ax2.set_title('Daily Gaming Hours → Addiction Risk\nDirect vs. Indirect Pathways', fontsize=10)
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('key_findings.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('Saved key_findings.png')

: 

: 

In [ ]:
# ── Narrative summary ────────────────────────────────────────────────────────
top_pred = addiction_paths.iloc[-1]  # largest |coef|
top_indirect_k = max(all_indirect, key=lambda k: abs(all_indirect[k]))
top_indirect_v = all_indirect[top_indirect_k]

addiction_r2 = results['gaming_addiction_risk_level'].rsquared

print('=' * 60)
print('KEY FINDINGS')
print('=' * 60)
print(f'\nModel R² (addiction risk): {addiction_r2:.3f}')
print(f'  → The model explains {addiction_r2*100:.1f}% of variance in addiction risk.')
print(f'\nStrongest direct predictor: {top_pred["from"]} (β={top_pred["coef"]:+.3f})')
print(f'\nDirect effect of daily gaming hours: β={direct:+.4f}')
print(f'Total indirect effect:               β={total_indirect:+.4f}')
print(f'Total effect:                        β={total_effect:+.4f}')
pct_direct = direct / total_effect * 100 if total_effect else 0
pct_indirect = total_indirect / total_effect * 100 if total_effect else 0
print(f'  → {pct_direct:.1f}% direct, {pct_indirect:.1f}% mediated through lifestyle factors')
print(f'\nLargest single indirect pathway:')
print(f'  gaming_hours → {top_indirect_k} → addiction_risk  (β={top_indirect_v:+.4f})')
print('=' * 60)

: 

: 